# Day 28: Advanced Topics — Hybrid Models & Modern Architectures

> **Goal:** Explore modern developments that bridge CNNs and Transformers: hybrid architectures, efficient attention, and the ConvNeXt/Swin Transformer approaches.

---

## The Convergence of CNNs and Transformers

The field has moved past "CNN vs ViT" toward designs that combine the best of both:

| Architecture | Key Idea | Year |
|---|---|---|
| **DeiT** | Data-efficient Image Transformer (distillation) | 2021 |
| **Swin Transformer** | Windowed attention (local → global) | 2021 |
| **ConvNeXt** | Modernize CNN with Transformer tricks | 2022 |
| **MaxViT** | Multi-axis attention + convolutions | 2022 |

The lesson: **inductive biases matter**, and the best architectures borrow ideas from both paradigms.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import math
import time

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Hybrid Architecture: CNN + Transformer

Use CNN for low-level feature extraction, then Transformer for global reasoning.

```
Image (3, 32, 32)
  → CNN backbone (conv blocks)     → (C, H', W')
  → Flatten to sequence            → (N, C)
  → Transformer Encoder            → (N, C)
  → Global Pool + Classify         → (num_classes)
```

In [ ]:
class HybridModel(nn.Module):
    """CNN backbone + Transformer encoder hybrid."""

    def __init__(self, num_classes=10, d_model=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()

        # CNN backbone — extracts local features
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, 2, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True),  # 32→16
            nn.Conv2d(128, d_model, 3, 2, 1, bias=False), nn.BatchNorm2d(d_model), nn.ReLU(inplace=True),  # 16→8
        )

        # Positional encoding for 8×8 = 64 spatial positions
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 65, d_model) * 0.02)  # 64+1

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=d_model*4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

        # Classification head
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B = x.shape[0]
        # CNN features
        x = self.backbone(x)                         # (B, d_model, 8, 8)
        x = x.flatten(2).transpose(1, 2)             # (B, 64, d_model)

        # Add CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)               # (B, 65, d_model)
        x = x + self.pos_embed

        # Transformer
        x = self.norm(self.encoder(x))
        return self.head(x[:, 0])                     # CLS token output


hybrid = HybridModel().to(device)
print(f"Hybrid model parameters: {sum(p.numel() for p in hybrid.parameters()):,}")

x = torch.randn(2, 3, 32, 32).to(device)
print(f"Input: {x.shape} → Output: {hybrid(x).shape}")

---
## 2. ConvNeXt-Style Blocks

ConvNeXt showed that **modernizing CNNs** with Transformer-era tricks makes them competitive with ViTs:

1. **Depthwise convolution** (like multi-head attention's per-head operation)
2. **Inverted bottleneck** (expand → contract, like FFN)
3. **LayerNorm** instead of BatchNorm
4. **GELU** instead of ReLU
5. **Fewer activation functions** (only one per block)

In [ ]:
class ConvNeXtBlock(nn.Module):
    """A ConvNeXt-style block: Depthwise Conv → LayerNorm → Expand → GELU → Contract."""

    def __init__(self, dim, expansion=4):
        super().__init__()
        # Depthwise conv: each channel has its own filter
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim)
        self.pwconv1 = nn.Linear(dim, dim * expansion)   # Expand
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(dim * expansion, dim)    # Contract

    def forward(self, x):
        residual = x                              # (B, C, H, W)
        x = self.dwconv(x)                        # (B, C, H, W) — depthwise
        x = x.permute(0, 2, 3, 1)                # (B, H, W, C) — for LayerNorm
        x = self.norm(x)
        x = self.pwconv1(x)                       # (B, H, W, 4C)
        x = self.act(x)
        x = self.pwconv2(x)                       # (B, H, W, C)
        x = x.permute(0, 3, 1, 2)                # (B, C, H, W)
        return x + residual


class MiniConvNeXt(nn.Module):
    """A small ConvNeXt-style network for CIFAR-10."""

    def __init__(self, num_classes=10, dims=(64, 128, 256)):
        super().__init__()

        # Stem: patchify with 4×4 conv (like ViT's patch embedding!)
        self.stem = nn.Sequential(
            nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
            nn.BatchNorm2d(dims[0]),  # using BN in stem for simplicity
        )

        # Stages
        self.stage1 = nn.Sequential(ConvNeXtBlock(dims[0]), ConvNeXtBlock(dims[0]))

        self.down1 = nn.Sequential(
            nn.Conv2d(dims[0], dims[1], 2, 2), nn.LayerNorm([dims[1], 4, 4]))
        self.stage2 = nn.Sequential(ConvNeXtBlock(dims[1]), ConvNeXtBlock(dims[1]))

        self.down2 = nn.Sequential(
            nn.Conv2d(dims[1], dims[2], 2, 2), nn.LayerNorm([dims[2], 2, 2]))
        self.stage3 = nn.Sequential(ConvNeXtBlock(dims[2]), ConvNeXtBlock(dims[2]))

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.LayerNorm(dims[2]), nn.Linear(dims[2], num_classes))

    def forward(self, x):
        x = self.stem(x)       # (B, 64, 8, 8)
        x = self.stage1(x)
        x = self.down1(x)      # (B, 128, 4, 4)
        x = self.stage2(x)
        x = self.down2(x)      # (B, 256, 2, 2)
        x = self.stage3(x)
        return self.head(x)


convnext = MiniConvNeXt().to(device)
print(f"MiniConvNeXt parameters: {sum(p.numel() for p in convnext.parameters()):,}")

x = torch.randn(2, 3, 32, 32).to(device)
print(f"Input: {x.shape} → Output: {convnext(x).shape}")

---
## 3. Comparison: Standard CNN vs Hybrid vs ConvNeXt

In [ ]:
# Data
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.2)])
test_transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])

full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)
train_data, val_data = random_split(full_train, [45000, 5000], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=128, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=128, shuffle=False)

### Data & Training Setup

We'll train the **Hybrid** and **MiniConvNeXt** models on CIFAR-10 with the same data augmentation and training recipe. This lets us see how modern architectural ideas translate into actual performance differences on a small dataset.

In [ ]:
# Training utilities
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

def train_and_evaluate(model, name, epochs=30, lr=1e-3):
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"val_acc": [], "time": []}
    for epoch in range(epochs):
        t0 = time.time()
        train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        vl, va = evaluate(model, val_loader, loss_fn, device)
        scheduler.step()
        history["val_acc"].append(va)
        history["time"].append(time.time() - t0)
        if (epoch+1) % 10 == 0:
            print(f"  {name} Epoch {epoch+1}: Val Acc = {va:.2%}")

    tl, ta = evaluate(model, test_loader, loss_fn, device)
    print(f"  {name} ✅ Test: {ta:.2%}")
    history["test_acc"] = ta
    return history

### Training Utilities

These helper functions are the same training and evaluation loops we've been building throughout the course. Both models use:
- **AdamW** optimizer with weight decay (0.05) and label smoothing (0.1)
- **Cosine annealing** learning rate schedule
- **30 epochs** of training

Using identical training infrastructure ensures a fair comparison.

In [ ]:
# Train all three
print("Training Hybrid (CNN+Transformer)...")
torch.manual_seed(42)
hybrid = HybridModel().to(device)
h_hybrid = train_and_evaluate(hybrid, "Hybrid", epochs=30)

print("\nTraining MiniConvNeXt...")
torch.manual_seed(42)
convnext = MiniConvNeXt().to(device)
h_convnext = train_and_evaluate(convnext, "ConvNeXt", epochs=30)

### Training the Models

We train both architectures and track validation accuracy over time. The **Hybrid** model benefits from CNN's efficient local feature extraction combined with the Transformer's global reasoning. The **ConvNeXt** shows that a purely convolutional design, when modernized with Transformer-era tricks, can be equally competitive.

In [ ]:
# Compare
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(h_hybrid["val_acc"], label=f"Hybrid ({sum(p.numel() for p in HybridModel().parameters()):,} params)")
ax1.plot(h_convnext["val_acc"], label=f"ConvNeXt ({sum(p.numel() for p in MiniConvNeXt().parameters()):,} params)")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Val Accuracy")
ax1.set_title("Validation Accuracy"); ax1.legend(); ax1.grid(True, alpha=0.3)

models_info = {
    "Hybrid": (sum(p.numel() for p in HybridModel().parameters()), h_hybrid["test_acc"]),
    "ConvNeXt": (sum(p.numel() for p in MiniConvNeXt().parameters()), h_convnext["test_acc"]),
}
names = list(models_info.keys())
test_accs = [v[1] for v in models_info.values()]
ax2.barh(names, test_accs, color=["C0", "C1"])
ax2.set_xlabel("Test Accuracy"); ax2.set_title("Test Accuracy Comparison")
ax2.set_xlim(min(test_accs) - 0.05, max(test_accs) + 0.02)
for i, (n, (p, a)) in enumerate(models_info.items()):
    ax2.text(a + 0.005, i, f"{a:.2%}\n({p:,} params)", va="center", fontsize=9)
ax2.grid(True, alpha=0.3, axis="x")

plt.tight_layout(); plt.show()

### Comparing the Results

The left plot shows how validation accuracy evolves during training — does one architecture converge faster? The right bar chart shows final test accuracy along with parameter counts. On a small dataset like CIFAR-10, the differences may be modest, but on larger datasets like ImageNet, these architectural choices have a much bigger impact.

---
## 4. Key Design Principles from Modern Architectures

In [ ]:
# Demonstrate depthwise separable convolution — key efficiency trick
print("=== Depthwise Separable Convolution ===\n")

in_ch, out_ch = 64, 128

# Standard convolution
standard_conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
standard_params = sum(p.numel() for p in standard_conv.parameters())

# Depthwise separable: depthwise (per-channel) + pointwise (1×1)
depthwise = nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1, groups=in_ch)
pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1)
sep_params = sum(p.numel() for p in depthwise.parameters()) + sum(p.numel() for p in pointwise.parameters())

print(f"Standard Conv2d({in_ch}, {out_ch}, 3): {standard_params:,} params")
print(f"Depthwise + Pointwise:                 {sep_params:,} params")
print(f"Reduction: {standard_params/sep_params:.1f}× fewer parameters!")

# Verify both produce same output shape
x = torch.randn(1, in_ch, 8, 8)
standard_out = standard_conv(x)
sep_out = pointwise(depthwise(x))
print(f"\nBoth produce: {standard_out.shape}")

### Depthwise Separable Convolution — The Key Efficiency Trick

A standard convolution applies a 3D filter across all input channels simultaneously. **Depthwise separable convolution** splits this into two steps:

1. **Depthwise conv:** Apply a separate 2D filter to each input channel independently (spatial filtering)
2. **Pointwise conv (1×1):** Mix information across channels (channel mixing)

This factorization dramatically reduces parameters while achieving similar representational power. It's used in MobileNets, EfficientNets, and is conceptually similar to how multi-head attention operates per-head.

---
## 🧪 Exercises

### Exercise 1: Windowed attention
Implement a simple "windowed" self-attention: divide the 8×8 feature map into 4×4 windows and apply attention only within each window. How many fewer FLOPs?

### Exercise 2: More ConvNeXt blocks
Add more blocks to each stage (3 instead of 2). Does it improve accuracy?

### Exercise 3: Hybrid variations
Try the reverse hybrid: start with Transformer layers, then use CNN layers. Does it work?

### Exercise 4: MobileNet-style blocks
Implement an inverted residual block (from MobileNetV2): 1×1 expand → depthwise 3×3 → 1×1 project. Build a small model with it.

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **Hybrid models** | CNN backbone + Transformer head — best of both worlds |
| **ConvNeXt** | CNNs modernized with Transformer tricks (LN, GELU, large kernels) |
| **Depthwise conv** | Per-channel convolution — efficient like attention per head |
| **Inverted bottleneck** | Expand → process → contract (like Transformer FFN) |
| **Design convergence** | Modern CNNs and Transformers share many design choices |

**Tomorrow:** Training Toolkit — learning rate schedules, early stopping, mixed precision, and other practical techniques!